# MATEN/T MiniAI v1 — Complete GPU training

**BY MATEN/T — M/ License v1.0**

This notebook trains the 86,045,328-parameter model with 6,144-token maximum context and RoPE positional encoding. Training is not artificially stopped at 100 steps. The full dataset is synthetic; an entire epoch is not a quality guarantee. Colab sessions can disconnect before completion.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU to use the CUDA training workflow.'
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))


## 1. Install project dependencies

The source is cloned directly from the official GitHub repository. Check the GPU and disk allocation before generating the complete corpus.


In [ ]:
from pathlib import Path
import subprocess, sys
PROJECT=Path('/content/MATEN-T-MiniAI')
if not (PROJECT/'miniai.py').exists():
    subprocess.run(['git','clone','https://github.com/opentaiofficial/MATEN-T-MiniAI.git',str(PROJECT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')],check=True)
subprocess.run([sys.executable,str(PROJECT/'miniai.py'),'info'],cwd=PROJECT,check=True)


## 2. Prepare all training records and tokenizer

A full generated corpus contains 9,400,000 JSONL records, of which 8,460,000 records form the training split per epoch. Run preparation only when the dataset or tokenizer is missing. Preparing a fresh corpus can consume gigabytes of disk space and substantial CPU time. Do not rebuild the tokenizer when resuming an existing checkpoint.


In [ ]:
import sys, subprocess
from pathlib import Path
DATA=PROJECT/'datasets'
TOK=PROJECT/'tokenizer'/'tokenizer.model'
from importlib.machinery import SourceFileLoader
import json
manifest=json.loads((DATA/'manifest.json').read_text(encoding='utf-8'))
domains=manifest['domains']
assert len(domains)==94, f'Expected 94 data domains, got {len(domains)}'
if not all((DATA/(domain+'.jsonl')).exists() for domain in domains):
    subprocess.run([sys.executable,str(PROJECT/'build_datasets.py'),'--count','100000'],cwd=PROJECT,check=True)
if not TOK.exists():
    subprocess.run([sys.executable,str(PROJECT/'miniai.py'),'prepare','--samples-per-domain','1000'],cwd=PROJECT,check=True)
print('Training data and tokenizer are ready.')


## 3. Train a complete epoch, with checkpoint recovery

A normal run has **no --max-steps cap**. Short examples are packed into dense blocks, and the model records actual tokens processed. The model's maximum context is **6,144**, but the default training window below is 1,024 for safer GPU memory use. You can select 6,144 if your GPU can accommodate it. Save checkpoints on Google Drive to survive Colab resets.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT=Path('/content/drive/MyDrive/MATEN-T-MiniAI-6K-checkpoints')
OUTPUT.mkdir(parents=True,exist_ok=True)
EPOCHS=1
SEQUENCE_LENGTH=1024  # Set to 6144 only if GPU memory is sufficient.
command=[sys.executable,str(PROJECT/'miniai.py'),'train','--device','cuda',
         '--precision','fp16','--epochs',str(EPOCHS),'--batch-size','1',
         '--grad-accum','8','--sequence-length',str(SEQUENCE_LENGTH),
         '--save-every','200','--log-every','20','--output',str(OUTPUT)]
checkpoint=OUTPUT/'last.pt'
if checkpoint.exists():
    command += ['--resume',str(checkpoint)]
    print('Resuming from:',checkpoint)
else:
    print('Starting a fresh complete-epoch training run.')
subprocess.run(command,cwd=PROJECT,check=True)


## 4. Evaluate the current weights

Review validation loss and manually inspect generated answers. Synthetic-template validation is not independent proof of language understanding. To train another complete epoch later, increase EPOCHS and rerun the training cell with the same model and tokenizer.


In [ ]:
subprocess.run([sys.executable,str(PROJECT/'miniai.py'),'evaluate',
                '--checkpoint',str(OUTPUT/'last.pt'),'--device','cuda',
                '--precision','fp16','--max-batches','20'],cwd=PROJECT,check=True)
